In [1]:
#Task 1: Random forest with default parameters on Iris
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
xtr, xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=0.3, random_state=0)

rf = RandomForestClassifier(random_state=42)
rf.fit(xtr, ytr)
print("Test accuracy:", round(accuracy_score(yte, rf.predict(xte)), 3))

Test accuracy: 0.978


In [2]:
#Task 2: Flipkart-style product category classifier with 20 products
prod = pd.DataFrame({
    "price": [45000, 52000, 1200, 800, 35000, 600, 1500, 900, 48000, 2500,
              700, 30000, 1100, 55000, 450, 1800, 38000, 650, 950, 42000],
    "rating": [4.5, 4.6, 4.0, 3.8, 4.4, 4.1, 4.2, 3.9, 4.7, 4.3,
               3.7, 4.2, 4.0, 4.8, 3.9, 4.4, 4.3, 3.8, 4.1, 4.5],
    "brand": ["Apple", "Samsung", "Puma", "Nike", "Samsung", "Puma", "Nike", "Puma", "Apple", "Nike",
              "Puma", "Samsung", "Nike", "Apple", "Puma", "Nike", "Samsung", "Puma", "Nike", "Apple"],
    "category": ["Mobile", "Mobile", "Shoes", "Shoes", "Mobile", "Shoes", "Shoes", "Shoes", "Mobile", "Shoes",
                 "Shoes", "Mobile", "Shoes", "Mobile", "Shoes", "Shoes", "Mobile", "Shoes", "Shoes", "Mobile"],
})

X = pd.get_dummies(prod[["price", "rating", "brand"]], columns=["brand"], dtype=int)
y = prod["category"]

rf2 = RandomForestClassifier(n_estimators=100, random_state=1)
rf2.fit(X, y)

print("Feature importance:")
print(pd.Series(rf2.feature_importances_, index=X.columns).sort_values(ascending=False).round(3))

Feature importance:
price            0.449
rating           0.249
brand_Apple      0.094
brand_Nike       0.086
brand_Samsung    0.068
brand_Puma       0.054
dtype: float64


In [3]:
#Task 3: Tune n_estimators and max_depth on Iris
best = 0
best_combo = None
for n in [5, 50, 200]:
    for d in [1, 3, 5]:
        m = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        m.fit(xtr, ytr)
        acc = accuracy_score(yte, m.predict(xte))
        print("n_estimators =", n, "max_depth =", d, "-> accuracy:", round(acc, 3))
        if acc > best:
            best = acc
            best_combo = (n, d)

print("\nBest combination (n_estimators, max_depth):", best_combo, "with accuracy", round(best, 3))

# Several combinations give the best accuracy of 0.978 (for example n_estimators 5 with max_depth 3 or 5, and 50 or 200 trees with max_depth 3 or 5).
# The first one found is (5, 3). max_depth 1 is too shallow and gives lower accuracy (0.667 for 50 trees).
# More trees did not help here because Iris is small and easy.

n_estimators = 5 max_depth = 1 -> accuracy: 0.956
n_estimators = 5 max_depth = 3 -> accuracy: 0.978
n_estimators = 5 max_depth = 5 -> accuracy: 0.978
n_estimators = 50 max_depth = 1 -> accuracy: 0.667
n_estimators = 50 max_depth = 3 -> accuracy: 0.978
n_estimators = 50 max_depth = 5 -> accuracy: 0.978


n_estimators = 200 max_depth = 1 -> accuracy: 0.956
n_estimators = 200 max_depth = 3 -> accuracy: 0.978


n_estimators = 200 max_depth = 5 -> accuracy: 0.978

Best combination (n_estimators, max_depth): (5, 3) with accuracy 0.978


In [4]:
#Task 4: Single decision tree vs random forest (bagging helps avoid overfitting)
from sklearn.datasets import make_classification

# a noisy dataset, where a single tree overfits easily
Xn, yn = make_classification(n_samples=600, n_features=20, n_informative=5, flip_y=0.2, random_state=1)
xtr, xte, ytr, yte = train_test_split(Xn, yn, test_size=0.3, random_state=1)

dt = DecisionTreeClassifier(random_state=1)
dt.fit(xtr, ytr)
rf3 = RandomForestClassifier(n_estimators=100, random_state=1)
rf3.fit(xtr, ytr)

print("Decision Tree -> train:", round(dt.score(xtr, ytr), 3), "test:", round(dt.score(xte, yte), 3))
print("Random Forest -> train:", round(rf3.score(xtr, ytr), 3), "test:", round(rf3.score(xte, yte), 3))

# The single tree learns the training data fully (train 1.0) but drops to 0.689 on test data, so it overfits to the noise.
# The random forest also has train 1.0 but its test accuracy is higher (0.794).
# Bagging trains each tree on a random sample of rows and features and then takes the vote of all trees,
# so the mistakes caused by noise in one tree are cancelled out by the others. This lowers variance and reduces overfitting.

Decision Tree -> train: 1.0 test: 0.689
Random Forest -> train: 1.0 test: 0.794


In [5]:
#Task 5: Random forest to predict if a Spotify song is a hit
sp = pd.read_csv("spotify_hits.csv")
X = sp[["tempo", "danceability", "energy"]]
y = sp["hit"]
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

rf4 = RandomForestClassifier(n_estimators=100, random_state=1)
rf4.fit(xtr, ytr)
print("Test accuracy:", round(rf4.score(xte, yte), 3))
print(pd.Series(rf4.feature_importances_, index=X.columns).round(3))

# One thing I noticed: the three features have almost equal importance (about 0.31 to 0.35),
# so no single feature decides if a song becomes a hit. The accuracy must be checked on test data (0.867), not on train data.

Test accuracy: 0.867
tempo           0.351
danceability    0.313
energy          0.336
dtype: float64
